In [254]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from src import synthetic

In [255]:
FORECAST_HORIZON = 10
NUM_STORES = 4
NUM_DAYS = 365*4

stores = synthetic.store_data(n_stores=NUM_STORES)
sales = synthetic.sales_data(n_stores=NUM_STORES, days=NUM_DAYS)

store1 = sales[sales['Store'] == 1]
sales = sales.loc[store1.index].set_index(['Date'])['Sales'].sort_index(ascending=True) # Sort from earliest to latest date

sales.loc[sales == 0] = np.nan
sales.ffill(inplace=True)  # Fill NaN values with the last valid observation


# Generate sales features

In [256]:
from src.features.lags import make_lags

def make_name(lag: pd.DateOffset, prefix: str) -> str:
    key   = list(lag.kwds.keys())[0]
    value = list(lag.kwds.values())[0]
    return f'{prefix}_{key}_{value}'


def make_lags(target: pd.Series, lags: list[pd.DateOffset]) -> pd.DataFrame:
    """ 
    Create lagged features for a given target series.

    Args:
        target (pd.Series): The target series for which to create lagged features. The index of the series should be a datetime index.
        lags (list[pd.DateOffset]): A list of pandas DateOffset objects representing the lags to create.

    Returns:
        pd.DataFrame: A DataFrame containing the lagged features, with each column named according to the lag applied (e.g., 'lag_days_1', 'lag_days_2', etc.).
    """
    
    # Ensure chronological order of the target series from earliest to latest date
    target = target.sort_index(ascending=True)
    
    x = []
    for lag in lags:
        sales_lag = target.shift(freq=lag)
        sales_lag.name = make_name(lag, prefix='lag')
        x.append(sales_lag)

    x = pd.concat(x, axis=1).reindex(target.index)

    return x


def make_diffs(target: pd.Series, diffs: list[pd.DateOffset], lag: pd.DateOffset) -> pd.DataFrame:
    """ 
    Create differenced features for a given target series.
    This function calculates the difference between the current value and the value at a specified lag (y[t]-y[t-lag]).
    During inference y[t] is unknown, so it's best to use lagged  target values to calculate the difference (y[t-lag]-y[t-2*lag]).
    This is why we use the lag parameter to shift the target series before calculating the difference.

    Args:
        target (pd.Series): The target series for which to create differenced features. The index of the series should be a datetime index.
        diffs (list[pd.DateOffset]): A list of pandas DateOffset objects representing the differences to create.
        lag (pd.DateOffset): A pandas DateOffset object representing the lag to use for calculating the difference.

    Returns:
        pd.DataFrame: A DataFrame containing the differenced features, with each column named according to the 
        difference applied (e.g., 'diff_days_1', 'diff_days_2', etc.).
    """
    
    # Ensure chronological order of the target series from earliest to latest date
    # and shift the target series by the specified lag to align historical values with current values
    target_lagged = target.sort_index(ascending=True).shift(freq=lag) 
    
    x = []
    for diff in diffs:
        # Shift the index along the calendar timeline to align historical dates to the target dates
        # Because names might mismatch due to shift, we align on target.index
        historical_target = target_lagged.shift(freq=diff).reindex(target.index)
        
        # Calculate the difference (Current Target - Historical Target)
        sales_diff = target_lagged - historical_target

        sales_diff.name = make_name(diff, prefix='diff')
        x.append(sales_diff)

    x = pd.concat(x, axis=1)#.reindex(target.index)
    return x


def make_rolling_features(target: pd.Series, windows: list[int], func: str, lag: pd.DateOffset) -> pd.DataFrame:
    """ 
    Create rolling features for a given target series. This function calculates the rolling mean for the target series over specified window sizes.
    To avoid data leakage, the target series is shifted by the specified lag before calculating the rolling mean.

    Args:
        target (pd.Series): The target series for which to create rolling features. The index of the series should be a datetime index.
        windows (list[int]): A list of integers representing the window sizes for the rolling calculations.
        func (str): The aggregation function to apply to the rolling window (e.g., 'mean', 'sum', etc.).
        lag (pd.DateOffset): A pandas DateOffset object representing the lag to use for calculating the rolling features.

    Returns:
        pd.DataFrame: A DataFrame containing the rolling features, with each column named according to the window
        size applied (e.g., 'rolling_mean_3', 'rolling_mean_5', etc.).
    """
    
    # Ensure chronological order of the target series from earliest to latest date
    # and shift the target series by the specified lag to align historical values with current values
    target_lagged = target.sort_index(ascending=True).shift(freq=lag) 
    
    x = []
    for window in windows:
        rolling_feature = target_lagged.rolling(window=window).agg(func)
        rolling_feature.name = f'rolling_{func}_{window}'
        x.append(rolling_feature)

    x = pd.concat(x, axis=1).reindex(target.index)

    return x

In [258]:
diffs = [pd.DateOffset(days=i) for i in range(1, 10)]
lags = [pd.DateOffset(days=i) for i in range(1, 10)]
windows = ['7D', '14D', '30D']
sales_ = sales.sample(frac=1, random_state=42)

xlag = make_lags(sales_, lags)
xdiff = make_diffs(sales_, diffs, lag=pd.DateOffset(days=1))
xroll_mean = make_rolling_features(sales_, windows, func='mean', lag=pd.DateOffset(days=1))
xroll_std = make_rolling_features(sales_, windows, func='std', lag=pd.DateOffset(days=1))

gg = pd.concat([sales_, xlag, xdiff, xroll_mean, xroll_std], axis=1).sort_index(ascending=True).head(100)
gg

,Sales,lag_days_1,lag_days_2,lag_days_3,lag_days_4,lag_days_5,lag_days_6,lag_days_7,lag_days_8,lag_days_9,...,diff_days_6,diff_days_7,diff_days_8,diff_days_9,rolling_mean_7D,rolling_mean_14D,rolling_mean_30D,rolling_std_7D,rolling_std_14D,rolling_std_30D
Date,,,,,,,,,,,,,,,,,,,,,
2013-01-01,467.320508,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-02,467.820508,467.320508,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,467.320508,467.320508,467.320508,NaN,NaN,NaN
2013-01-03,451.000000,467.820508,467.320508,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,467.570508,467.570508,467.570508,0.353553,0.353553,0.353553
2013-01-04,434.179492,451.000000,467.820508,467.320508,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,462.047005,462.047005,462.047005,9.570253,9.570253,9.570253
2013-01-05,534.679492,434.179492,451.000000,467.820508,467.320508,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,455.080127,455.080127,455.080127,15.975275,15.975275,15.975275
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2013-04-06,480.179492,479.679492,496.500000,513.320508,512.820508,595.000000,576.679492,576.679492,476.179492,493.000000,...,-97.000000,3.5,-13.320508,-130.141016,535.811356,534.061356,520.129915,45.778981,55.294918,50.545359
2013-04-07,480.179492,480.179492,479.679492,496.500000,513.320508,512.820508,595.000000,576.679492,576.679492,476.179492,...,-96.500000,-96.5,4.000000,-12.820508,522.025642,534.561356,520.052565,45.950461,54.730822,50.606669
2013-04-08,598.500000,480.179492,480.179492,479.679492,496.500000,513.320508,512.820508,595.000000,576.679492,576.679492,...,-114.820508,-96.5,-96.500000,4.000000,508.239927,535.061356,520.535898,41.033329,54.155880,50.136516


In [262]:

sales = synthetic.sales_data(n_stores=NUM_STORES, days=NUM_DAYS)
sales_fixed = sales.set_index(['Date'])[['Store', 'Sales']].sort_index(ascending=True)
x = sales_fixed.groupby('Store').apply(lambda x: make_lags(x, lags), include_groups=False)
x

Sales        Sales        Sales        Sales  \
Store Date                                                             
1     2013-01-01          NaN          NaN          NaN          NaN   
      2013-01-02   467.320508          NaN          NaN          NaN   
      2013-01-03   467.820508   467.320508          NaN          NaN   
      2013-01-04   451.000000   467.820508   467.320508          NaN   
      2013-01-05   434.179492   451.000000   467.820508   467.320508   
...                       ...          ...          ...          ...   
4     2016-12-26     0.000000  3788.679492  3786.679492  3802.000000   
      2016-12-27  3910.000000     0.000000  3788.679492  3786.679492   
      2016-12-28  3829.320508  3910.000000     0.000000  3788.679492   
      2016-12-29  3831.320508  3829.320508  3910.000000     0.000000   
      2016-12-30  3816.000000  3831.320508  3829.320508  3910.000000   

                        Sales        Sales        Sales        Sales  \
Store Date                                                             
1     2013-01-01          NaN          NaN          NaN          NaN   
      2013-01-02          NaN          NaN          NaN          NaN   
      2013-01-03          NaN          NaN          NaN          NaN   
      2013-01-04          NaN          NaN          NaN          NaN   
      2013-01-05          NaN          NaN          NaN          NaN   
...                       ...          ...          ...          ...   
4     2016-12-26  3817.320508  3815.320508  3796.000000     0.000000   
      2016-12-27  3802.000000  3817.320508  3815.320508  3796.000000   
      2016-12-28  3786.679492  3802.000000  3817.320508  3815.320508   
      2016-12-29  3788.679492  3786.679492  3802.000000  3817.320508   
      2016-12-30     0.000000  3788.679492  3786.679492  3802.000000   

                        Sales  
Store Date                     
1     2013-01-01          NaN  
      2013-01-02          NaN  
      2013-01-03          NaN  
      2013-01-04          NaN  
      2013-01-05          NaN  
...                       ...  
4     2016-12-26  3774.679492  
      2016-12-27     0.000000  
      2016-12-28  3796.000000  
      2016-12-29  3815.320508  
      2016-12-30  3817.320508  

[5840 rows x 9 columns]